# Breast-Micro-Calc preprocessing

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
os.chdir(REPO_ROOT)  
sys.path.insert(0, str(REPO_ROOT))

## Load raw data

In [ ]:
from preprocessing.breast_micro_calc import BreastMicroCalc, csv_save_path
from utils.preprocessing import birads_assessment, breast_density, get_value, laterality

ds = BreastMicroCalc()
ds.set_dataset_name("breast-micro-calc")
print(ds.normal_cases.shape, ds.suspicious_cases.shape)
ds.normal_cases.head()

### Step 1 — clean and rename normal-case columns, tag the subfolder

In [ ]:
from utils.preprocessing import csv_column_cleaning

ds.normal_cases.columns = csv_column_cleaning(list(ds.normal_cases.columns))
ds.normal_cases = ds.normal_cases.rename(columns={
    "bi-rads categories for breast density": "breast density",
    "bi-rads categories for classification": "birads",
    "breast right/left": "laterality",
    "age at the time of the recent mammogram": "age",
})
ds.normal_cases["subfolder"] = "Normal_cases"
ds.normal_cases.columns.tolist()

### Step 2 — same cleanup for suspicious-case columns

In [ ]:
ds.suspicious_cases.columns = csv_column_cleaning(list(ds.suspicious_cases.columns))
ds.suspicious_cases = ds.suspicious_cases.rename(columns={
    "bi-rads categories for breast density": "breast density",
    "bi-rads categories for classification": "birads",
    "breast right/left": "laterality",
    "age at the time of the recent mammogram": "age",
})
ds.suspicious_cases["subfolder"] = "Suspicious_cases"
ds.suspicious_cases.columns.tolist()

### Step 3 — combine both sheets and clear `NOT AVAILABLE` placeholders

In [ ]:
cases_df = pd.concat([ds.normal_cases, ds.suspicious_cases], axis=0, ignore_index=True)
cases_df = cases_df.replace("NOT AVAILABLE", None)
cases_df.shape

### Step 4 — map BI-RADS, laterality, and breast density to harmonized labels

In [ ]:
cases_df["birads"] = cases_df["birads"].apply(lambda x: get_value(x, birads_assessment))
cases_df["laterality"] = cases_df["laterality"].apply(lambda x: get_value(x, laterality))
cases_df["breast density"] = cases_df["breast density"].apply(lambda x: get_value(x, breast_density))
cases_df[["birads", "laterality", "breast density"]].apply(lambda c: c.value_counts(dropna=False))

## Build exam records for one case

In [ ]:
row = cases_df.iloc[0]
exams = ds.process_row(row)
exams

## Final processed CSV

In [ ]:
import json

final_df = pd.read_csv(csv_save_path)
print(f"cases before per-exam processing: {len(cases_df)} | rows in final csv: {len(final_df)}")
final_df.head()

In [ ]:
sample = final_df.iloc[0]
print(json.dumps(json.loads(sample['context']), indent=2))
print(json.dumps(json.loads(sample['findings']), indent=2))